# Pipeline RAG (Retrieval-Augmented Generation)

## Ce que tu vas apprendre
- Stratégies de recherche vectorielle (KNN, ANN) et leur évaluation.
- Utilité des bases de données vectorielles (recherche par similarité, RAG).
- Différences entre bases de données vectorielles, librairies et plugins.
- Bonnes pratiques d'utilisation et de performance des vector stores.
- Comment un modèle de langage exploite du contexte pour répondre à une question.
- Génération d'embeddings de texte et stockage vectoriel.
- Interrogation d'un vector store pour récupérer des documents pertinents.
- Utilisation d'un modèle de langage pour répondre à des questions avec du contexte récupéré.

## Ce que tu vas créer
Un pipeline RAG fonctionnel : vectorisation de texte, stockage vectoriel dans FAISS et ChromaDB, recherche par similarité, et question-réponse avec un modèle Hugging Face.

---

**Avant de commencer, quelques points que je ne vais pas te laisser passer sans les signaler :**

1. L'énoncé original te donne des commandes shell à taper "dans ton terminal". Ici je les mets dans des cellules `%pip` / `!` pour que le notebook soit auto-suffisant, mais l'esprit reste le même.
2. `pip install -q numpy<2` tel qu'écrit dans l'énoncé est **une commande shell invalide** — le `<` sera interprété par ton shell comme une redirection de fichier, pas comme une contrainte de version. La syntaxe correcte est `pip install -q "numpy<2"` (avec des guillemets). Je l'ai corrigée ci-dessous.
3. `chromadb==0.3.21` est une **version très ancienne** (2023). L'API a changé depuis (notamment `Client()`, `list_collections()`, `create_collection()`). Si tu installes une version récente de chromadb, une partie du code ci-dessous devra être adaptée. Je te le signale plutôt que de te laisser découvrir une erreur cryptique plus tard.
4. Le chemin vers `labelled_newscatcher_dataset.csv` n'est précisé nulle part dans l'énoncé. Je mets un chemin par défaut (`./data/labelled_newscatcher_dataset.csv`) — **à toi de l'adapter** à l'endroit réel où tu as téléchargé le fichier.
5. Utiliser `gpt2` pour la génération de réponse (étape 5) va produire des réponses de qualité très moyenne, voire incohérentes — GPT-2 n'est pas instruction-tuned, il ne "répond" pas vraiment à une question, il continue le texte statistiquement. Ne sois pas surpris si le résultat final ressemble à du remplissage plutôt qu'à une vraie réponse. C'est une limite du modèle choisi dans l'exercice, pas une erreur de ta part.

## Étape 0 : Installation des librairies

In [ ]:
%pip install -q faiss-cpu==1.7.4
%pip install -q chromadb
%pip install -q "numpy<2"
%pip install -q sentence-transformers transformers accelerate pandas

In [ ]:
import os
os.makedirs("cache", exist_ok=True)
print("Dossier cache prêt.")

# Sous Linux (Colab / Ubuntu), libomp est nécessaire pour faiss :
# !apt install -y libomp-dev
# python -m pip install --upgrade faiss-cpu

In [ ]:
import numpy as np
import pandas as pd
import faiss
import json
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

---
## Exercice 1 : Chargement et préparation des données

In [ ]:
# ⚠️ Adapte ce chemin à l'emplacement réel du fichier CSV sur ta machine
path = "./data/labelled_newscatcher_dataset.csv"

# Ce dataset est séparé par des ';' (spécificité connue de labelled_newscatcher_dataset.csv)
pdf = pd.read_csv(path, sep=";")

In [ ]:
# Un identifiant unique par ligne, utile pour la suite (FAISS, ChromaDB)
pdf["id"] = pdf.index

In [ ]:
display(pdf.head())
print(pdf.info())
print("Valeurs manquantes par colonne :")
print(pdf.isna().sum())

In [ ]:
# Sous-ensemble plus petit pour itérer rapidement pendant le développement
pdf_subset = pdf.head(1000).reset_index(drop=True)
print(f"Taille du sous-ensemble : {len(pdf_subset)}")

---
## Exercice 2 : Vectorisation avec Sentence Transformers

In [ ]:
def example_create_fn(doc1: pd.Series) -> InputExample:
    """
    Fonction utilitaire qui transforme une ligne (un titre) en objet InputExample
    attendu par sentence-transformers.
    """
    return InputExample(texts=[doc1])

In [ ]:
faiss_train_examples = pdf_subset.apply(lambda x: example_create_fn(x["title"]), axis=1).tolist()
faiss_train_examples[:10]

In [ ]:
from sentence_transformers import SentenceTransformer

# Modèle léger et rapide, bon compromis qualité/vitesse pour de l'embedding de phrases courtes
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
titles_list = pdf_subset["title"].tolist()

In [ ]:
faiss_title_embedding = model.encode(titles_list, show_progress_bar=True)

In [ ]:
print(len(faiss_title_embedding), len(faiss_title_embedding[0]))
# Tu devrais voir : (1000, 384) -> 1000 titres, chacun encodé sur 384 dimensions
# (384 = la dimension native du modèle all-MiniLM-L6-v2)

---
## Exercice 3 : Indexation et recherche avec FAISS

In [ ]:
import numpy as np
import faiss

In [ ]:
pdf_to_index = pdf_subset.set_index("id", drop=False)
id_index = np.array(pdf_to_index["id"]).astype("int64")

In [ ]:
# FAISS attend un tableau numpy en float32, pas une liste Python
content_encoded_normalized = np.array(faiss_title_embedding, dtype="float32").copy()
faiss.normalize_L2(content_encoded_normalized)

In [ ]:
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(len(faiss_title_embedding[0])))
index_content.add_with_ids(content_encoded_normalized, id_index)

In [ ]:
def search_content(query, pdf_to_index, k=3):
    query_vector = model.encode([query]).astype("float32")
    faiss.normalize_L2(query_vector)

    top_k = index_content.search(query_vector, k)
    ids = top_k[1][0].tolist()
    similarities = top_k[0][0].tolist()

    results = pdf_to_index.loc[ids].copy()
    results["similarities"] = similarities
    return results

In [ ]:
display(search_content("animal", pdf_to_index, k=5))

---
## Exercice 4 : Collection et requêtes avec ChromaDB

⚠️ Comme signalé en introduction : ce code correspond à l'API de **chromadb 0.3.x**. Si tu as installé une version plus récente (>=0.4), `chroma_client.list_collections()` ne renvoie plus des objets qu'on compare directement de la même façon, et `chroma_client.get_or_create_collection()` remplace avantageusement ce bricolage de suppression manuelle.

In [ ]:
import chromadb
from chromadb.config import Settings

In [ ]:
chroma_client = chromadb.Client()
collection_name = "my_news"

existing_collections = [c.name for c in chroma_client.list_collections()]
if collection_name in existing_collections:
    chroma_client.delete_collection(name=collection_name)

print(f"Creating collection: '{collection_name}'")
collection = chroma_client.create_collection(name=collection_name)

In [ ]:
display(pdf_subset)

collection.add(
    documents=pdf_subset["title"][:100].tolist(),
    metadatas=[{"topic": topic} for topic in pdf_subset["topic"][:100].tolist()],
    ids=[str(i) for i in pdf_subset["id"][:100].tolist()]
)

In [ ]:
import json

results = collection.query(
    query_texts=["space"],
    n_results=10
)

print(json.dumps(results, indent=4))

---
## Exercice 5 : Question-réponse avec un modèle Hugging Face

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [ ]:
model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForCausalLM.from_pretrained(model_id)

In [ ]:
pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    device_map="auto",  # nécessite le package `accelerate` installé plus haut
)

In [ ]:
question = "What's the latest news on space development?"
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])
prompt_template = f"Relevant context: {context}\n\n The user's question: {question}"

print(prompt_template)

In [ ]:
lm_response = pipe(prompt_template)
print(lm_response[0]["generated_text"])

---
## Bilan honnête

Ce que tu viens de construire est un **RAG minimal**, utile pour comprendre les briques (embedding → index vectoriel → recherche → prompt → génération), mais ce n'est **pas** un système de production :

- **GPT-2 n'est pas fait pour répondre à des questions.** Il continue statistiquement le texte du prompt. Pour un vrai Q&A, utilise un modèle instruction-tuned (ex. `mistralai/Mistral-7B-Instruct`, ou un modèle plus petit comme `google/flan-t5-base` qui, lui, est entraîné pour suivre des instructions).
- **Le contexte n'est jamais nettoyé ni limité en taille.** Si les documents récupérés sont longs, tu peux dépasser la fenêtre de contexte du modèle sans avertissement clair.
- **Aucune évaluation de la pertinence du RAG n'est faite.** Tu n'as aucune métrique pour savoir si les documents récupérés sont réellement utiles à la réponse — juste un score de similarité cosinus, qui ne mesure pas la pertinence sémantique pour répondre à la question.
- **FAISS et ChromaDB font ici doublon** : l'exercice te fait implémenter la même chose deux fois avec deux outils différents, sans jamais comparer leurs performances ou leurs cas d'usage respectifs (FAISS = librairie bas niveau très rapide, sans persistance ni métadonnées native ; ChromaDB = base de données complète avec persistance et métadonnées, mais plus lente et plus lourde). Vaut le coup de creuser ça toi-même plutôt que de considérer que "les deux font la même chose".